In [ ]:
import cdsapi

    
# Create a client instance
client = cdsapi.Client()

# Dataset details
dataset = "derived-era5-single-levels-daily-statistics"
variables = {
    "product_type": "reanalysis",
    "variable": ["volumetric_soil_water_layer_1"], # download all the data by changing the variables here
    "month": [
        "01", "02", "03",
        "09", "10", "11",
        "12"],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "daily_statistic": "daily_mean",
    "time_zone": "utc+00:00",
    "frequency": "1_hourly",
    "area": [-60, -180, -90, 180]
}

# Iterate over each year and send a request
for year in range(2005, 2025):  # Update the range as necessary
    # Update the year in the request dictionary
    variables['year'] = [str(year)]
    
    # Define the output file name for each year
    output_file = f'G:/Hangkai/Anttarctic Vegetation Dynamic/ERA5_Climate_Data/volumetric_soil_water_layer_1_{year}.zip'
    
    # Retrieve and download the data for the specific year
    client.retrieve(dataset, variables).download(output_file)
    
    # Print the status message
    print(f'Data for the year {year} downloaded and saved as {output_file}')

In [ ]:
import cdsapi

    
# Create a client instance
client = cdsapi.Client()

# Dataset details
dataset = "derived-era5-pressure-levels-daily-statistics"
variables = {
    "product_type": "reanalysis",
    "variable": ["relative_humidity"],
    "month": [
        "01", "02", "03",
        "09", "10", "11",
        "12"
    ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "pressure_level": ["1000"],
    "daily_statistic": "daily_mean",
    "time_zone": "utc+00:00",
    "frequency": "1_hourly",
    "area": [-60, -180, -90, 180]
}

# Iterate over each year and send a request
for year in range(2015, 2016):  # Update the range as necessary
    # Update the year in the request dictionary
    variables['year'] = [str(year)]
    
    # Define the output file name for each year
    output_file = f'G:/Hangkai/Anttarctic Vegetation Dynamic/ERA5_Climate_Data/relative_humidity_{year}.nc'
    
    # Retrieve and download the data for the specific year
    client.retrieve(dataset, variables).download(output_file)
    
    # Print the status message
    print(f'Data for the year {year} downloaded and saved as {output_file}')

In [ ]:
import os
import netCDF4 as nc
import numpy as np
import rasterio
from rasterio.transform import from_origin
import zipfile
import tempfile
from datetime import datetime

def date_in_range(year, month, day):
    start_date = datetime(year, 9, 23)
    end_date = datetime(year + 1, 3, 21)
    current_date = datetime(year, month, day)
    if current_date >= start_date:
        return True
    if current_date <= end_date and current_date.year == year + 1:
        return True
    return False

folder_path = r'G:\Hangkai\Anttarctic Vegetation Dynamic\ERA5_Climate_Data'
output_folder_path = r'G:\Hangkai\Anttarctic Vegetation Dynamic\ERA5_Climate_Data\2m_temperature'
years = range(2002, 2024)  # From 2002 to 2023

for year in years:
    zip_path = os.path.join(folder_path, f'2m_temperature_{year}.zip')
    sum_precip = None
    count = 0

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_name in zip_ref.namelist():
            if file_name.endswith('.nc'):
                date_str = file_name.split('_')[-2]
                date_obj = datetime.strptime(date_str, "%Y%m%d")
                if date_in_range(year, date_obj.month, date_obj.day):
                    with tempfile.TemporaryDirectory() as tmpdirname:
                        local_nc_path = os.path.join(tmpdirname, file_name)
                        zip_ref.extract(file_name, tmpdirname)
                        with nc.Dataset(local_nc_path) as ds:
                            precip_data = ds.variables['Temperature_Air_2m_Mean_24h'][:]
                            fill_value = ds.variables['Temperature_Air_2m_Mean_24h']._FillValue
                            precip_data = np.where(precip_data == fill_value, np.nan, precip_data)
                            if sum_precip is None:
                                sum_precip = np.zeros_like(precip_data, dtype=float)
                            sum_precip += precip_data
                            count += 1

    if count > 0:
        average_precip = sum_precip / count
        average_precip = np.nan_to_num(average_precip, nan=0.0)
        average_precip = average_precip.squeeze()
        output_path = os.path.join(output_folder_path, f'average_2m_temperature_{year}.tif')

        # Check if the output directory exists, create if not
        os.makedirs(output_folder_path, exist_ok=True)

        with rasterio.open(output_path, 'w', driver='GTiff', height=average_precip.shape[0],
                           width=average_precip.shape[1], count=1, dtype=str(average_precip.dtype),
                           crs='+proj=latlong', transform=from_origin(-180, -60, 0.1, 0.1)) as dst:
            dst.write(average_precip, 1)

from osgeo import gdal

# Base file paths and template
template_path = "G:/Hangkai/Anttarctic Vegetation Dynamic/Version_2_data/Antarctica_Vegetation_Duration/First_last/First_2003.tif"
base_input_path = "G:/Hangkai/Anttarctic Vegetation Dynamic/ERA5_Climate_Data/2m_temperature/average_2m_temperature_{year}.tif"
base_output_path = "G:/Hangkai/Anttarctic Vegetation Dynamic/ERA5_Climate_Data/2m_temperature/2m_temperature_{year}.tif"

# Open template file
template_ds = gdal.Open(template_path, gdal.GA_ReadOnly)
template_proj = template_ds.GetProjection()
template_geotrans = template_ds.GetGeoTransform()
x_size = template_ds.RasterXSize
y_size = template_ds.RasterYSize

# Loop through the years
for year in range(2002, 2024):
    input_path = base_input_path.format(year=year)
    output_path = base_output_path.format(year=year)
    
    # Create output file
    driver = gdal.GetDriverByName('GTiff')
    output_ds = driver.Create(output_path, x_size, y_size, 1, gdal.GDT_Float32)
    output_ds.SetProjection(template_proj)
    output_ds.SetGeoTransform(template_geotrans)

    # Setup reprojection options using nearest neighbor interpolation
    warp_options = gdal.WarpOptions(
        srcSRS='EPSG:4326',  # Source coordinate system
        dstSRS=template_proj,  # Destination coordinate system
        resampleAlg=gdal.GRA_NearestNeighbour,
        outputBounds=[template_geotrans[0], template_geotrans[3] + y_size * template_geotrans[5], template_geotrans[0] + x_size * template_geotrans[1], template_geotrans[3]],
        width=x_size,
        height=y_size
    )

    # Execute reprojection
    gdal.Warp(output_ds, gdal.Open(input_path), options=warp_options)

    # Close datasets
    output_ds = None

    print("Reprojection complete for year {}. Output saved to: {}".format(year, output_path))

# Close template dataset
template_ds = None

from osgeo import gdal
import numpy as np
import rasterio

# Base output path from previous reprojection
base_output_path = "G:/Hangkai/Anttarctic Vegetation Dynamic/ERA5_Climate_Data/2m_temperature/2m_temperature_{year}.tif"
# Loop through the years
for year in range(2002, 2024):
    output_path = base_output_path.format(year=year)

    with rasterio.open(output_path) as src:
            # Read the first band
            band = src.read(1)
    
    # Replace no data values and NaN values with np.nan
    band[band == -9999] = np.nan
    
    # Compute mean, ignoring NaN values
    mean_value = np.nanmean(np.nanmean(band))
    
    # Output the result
    print("Mean value for year", year, " is: ", mean_value)

    # Close the dataset
    ds = None